In [1]:
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import random
import copy

2026-01-12 17:27:34.881555: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/qduong/Projects/MusicEditLearning/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
tfrecord_path = "data/test/*.tfrecord"


def parse_seq_example(example_proto):
    # Define how to parse
    context_features = {
    }

    sequence_features = {
        "pitch_seq": tf.io.VarLenFeature(dtype=tf.int64),
    }
    _, sequence = tf.io.parse_single_sequence_example(
        example_proto,
        context_features=context_features,
        sequence_features=sequence_features
    )
    pitch_seq = tf.sparse.to_dense(sequence["pitch_seq"])
    pitch_seq = tf.reshape(pitch_seq, [-1])
    return pitch_seq

# Create dataset
# dataset = tf.data.TFRecordDataset(tf.io.gfile.glob(tfrecord_path))
# dataset = dataset.map(parse_seq_example)





In [3]:
# Inspect first example
# for seq in dataset.take(1):
#     print(seq.numpy())

In [3]:
def pitch_seq_to_notes(pitch_seq):
    notes = []
    i = 0
    while i < len(pitch_seq):
        token = pitch_seq[i]
        if token == 128:
            # hold token without a preceding note — skip
            i += 1
            continue
        if token == 129:
            # rest — skip or treat as durationless rest
            i += 1
            continue

        # Otherwise, this is a real note onset
        pitch = int(token)
        duration = 1
        j = i + 1
        # Count holds
        while j < len(pitch_seq) and pitch_seq[j] == 128:
            duration += 1
            j += 1

        notes.append((pitch, duration))
        i = j

    return notes


In [4]:
# notes = pitch_seq_to_notes(seq.numpy())
# print(notes)

In [4]:
PITCH_MIN = 21
PITCH_MAX = 108
DURATIONS = [1, 2, 3, 4, 6, 8] 

SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>"]

pitch_tokens = [f"P{p}" for p in range(PITCH_MIN, PITCH_MAX + 1)]
duration_tokens = [f"D{d}" for d in DURATIONS]

vocab = SPECIAL_TOKENS + pitch_tokens + duration_tokens

token_to_id = {tok: i for i, tok in enumerate(vocab)}
id_to_token = {i: tok for tok, i in token_to_id.items()}

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]



In [5]:
def melody_is_valid(notes):
    return all(d in DURATIONS for _, d in notes)


def notes_to_tokens(notes, edited_note_indices=None):
    tokens = [BOS_ID]
    mask = [0]

    for i, (pitch, dur) in enumerate(notes):
        tokens.append(token_to_id[f"P{pitch}"])
        tokens.append(token_to_id[f"D{dur}"])

        is_edited = edited_note_indices and i in edited_note_indices
        mask.extend([1 if is_edited else 0, 1 if is_edited else 0])

    tokens.append(EOS_ID)
    mask.append(0)
    return tokens, mask



EDIT_TYPES = ["replace_pitch", "change_duration", "delete_note", "insert_note"]

def sample_edit_type():
    """Samples an edit type from EDIT_TYPES."""
    return random.choice(EDIT_TYPES)

def sample_position(melody, edit_type):
    """Samples a valid position for a given edit type."""
    if edit_type == "insert_note":
        # Can insert at the end
        return random.randint(0, len(melody))

    return random.randint(0, len(melody) - 1)

def sample_duration(exclude_duration):
    """Samples a duration from DURATIONS, excluding exclude_duration."""
    new_durations = [d for d in DURATIONS if d != exclude_duration]
    if not new_durations:
        return exclude_duration
    return random.choice(new_durations)

def sample_insert_note():
    """Samples a random note to insert."""
    pitch = random.randint(PITCH_MIN, PITCH_MAX)
    duration = random.choice(DURATIONS)
    return (pitch, duration)

def simulate_edits(melody, max_edits=5):
    melody_corrupt = copy.deepcopy(melody)
    edit_script = []

    n_edits = random.randint(1, max_edits)

    for _ in range(n_edits):
        edit_type = sample_edit_type()

        if edit_type == "insert_note":
            t = sample_position(melody_corrupt, edit_type)
            note = sample_insert_note()
            melody_corrupt.insert(t, note)
            edit_script.append(("delete_note", t))
        elif len(melody_corrupt) == 0: # No other edits possible on empty melody
            continue
        else:
            t = sample_position(melody_corrupt, edit_type)
            if edit_type == "replace_pitch":
                old_p, d = melody_corrupt[t]
                # Make sure new pitch is different and within bounds
                new_p = old_p
                while new_p == old_p:
                    new_p = old_p + random.choice([-2, -1, 1, 2])
                    new_p = max(PITCH_MIN, min(PITCH_MAX, new_p))

                melody_corrupt[t] = (new_p, d)
                edit_script.append(("replace_pitch", t, old_p))

            elif edit_type == "change_duration":
                p, old_d = melody_corrupt[t]
                new_d = sample_duration(old_d)
                melody_corrupt[t] = (p, new_d)
                edit_script.append(("change_duration", t, old_d))

            elif edit_type == "delete_note":
                note = melody_corrupt.pop(t)
                edit_script.append(("insert_note", t, note))

    return melody_corrupt, edit_script




In [6]:


def collate_fn(batch):
    corrupted, clean, masks, edit_scripts = zip(*batch)

    corrupted = pad_sequence(corrupted, batch_first=True, padding_value=PAD_ID)
    clean = pad_sequence(clean, batch_first=True, padding_value=PAD_ID)
    masks = pad_sequence(masks, batch_first=True, padding_value=0)

    # print((corrupted == PAD_ID).sum())
    # print((clean == PAD_ID).sum())

    return corrupted, clean, masks, edit_scripts


class MelodyDataset(Dataset):
    def __init__(self, tfrecord_path):
        self.dataset = tf.data.TFRecordDataset(tf.io.gfile.glob(tfrecord_path))
        self.dataset = self.dataset.map(parse_seq_example)
        self.samples = []

        for pitch_seq in self.dataset:
            notes = pitch_seq_to_notes(pitch_seq.numpy())
            if len(notes) > 0 and melody_is_valid(notes):
                self.samples.append(notes)

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        original_notes = self.samples[idx]
        corrupted_notes, edit_script = simulate_edits(original_notes)
        
        edited_positions = {e[1] for e in edit_script if len(e) > 1}

        corrupted_tokens, edit_mask = notes_to_tokens(corrupted_notes, edited_positions)
        original_tokens, _ = notes_to_tokens(original_notes)

        
        return (
            torch.tensor(corrupted_tokens, dtype=torch.long),
            torch.tensor(original_tokens, dtype=torch.long),
            torch.tensor(edit_mask, dtype=torch.long),
            edit_script
        )

In [7]:
dataset = MelodyDataset("data/test/*.tfrecord")
print(f"Dataset size: {len(dataset)}")

2026-01-12 17:27:45.888352: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-01-12 17:27:45.971233: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


Dataset size: 13957


2026-01-12 17:27:48.369947: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [10]:
dataset[3]

(tensor([ 1, 58, 92, 56, 92, 54, 92, 53, 92, 58, 94, 58, 94, 58, 96, 54, 94, 54,
         94, 56, 92, 54, 92, 53, 92, 51, 92, 68, 95, 51, 94, 51, 94, 53, 92, 51,
         92, 49, 92, 48, 92, 56, 94, 56, 94,  2]),
 tensor([ 1, 58, 92, 56, 92, 54, 92, 53, 92, 58, 94, 58, 94, 58, 96, 54, 94, 54,
         94, 56, 92, 54, 92, 53, 92, 51, 92, 51, 94, 51, 94, 53, 92, 51, 92, 49,
         92, 48, 92, 56, 94, 56, 94,  2]),
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 [('delete_note', 13)])

In [ ]:
# implement training pipeline here

In [8]:


# Create DataLoader
batch_size = 32
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

# Example of how to use the dataloader
# for corrupted_batch, clean_batch, masks_batch, edit_script_batch in train_dataloader:
#    print(f"Corrupted batch shape: {corrupted_batch.shape}")
#    print(f"Clean batch shape: {clean_batch.shape}")
#    print(f"Masks batch shape: {masks_batch.shape}")
#    print(f"Edit script batch (first item): {edit_script_batch[0]}")
#    break

In [9]:
def causal_mask(size, device):
    return torch.triu(
        torch.ones(size, size, device=device), diagonal=1
    ).bool()


class MelodyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_embed = nn.Embedding(512, d_model)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, n_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)

        mask = causal_mask(T, x.device)

        x = self.embed(x) + self.pos_embed(pos)
        x = self.decoder(x, x, tgt_mask=mask)
        return self.fc(x)

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
model = MelodyTransformer(vocab_size=len(vocab), d_model=256, n_heads=4, n_layers=4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
def train_baseline(
    model,
    dataloader,
    optimizer,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    epochs=10
):
    model.train()
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    for epoch in range(epochs):
        total_loss = 0.0

        for corrupted, clean, _, _ in dataloader:
            corrupted = corrupted.to(device)
            clean = clean.to(device)

            # Teacher forcing
            inputs = corrupted[:, :-1]
            targets = clean[:, 1:]

            T = min(inputs.size(1), targets.size(1))

            inputs = inputs[:, :T]
            targets = targets[:, :T]

            logits = model(inputs)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"[Baseline] Epoch {epoch+1}: loss = {avg_loss:.4f}")


In [58]:
batch = next(iter(train_dataloader))
print(batch[0].shape, batch[1].shape)


torch.Size([32, 122]) torch.Size([32, 120])


In [12]:
train_baseline(model, train_dataloader, optimizer, epochs=20)

[Baseline] Epoch 1: loss = 1.2288
[Baseline] Epoch 2: loss = 0.6923
[Baseline] Epoch 3: loss = 0.6581
[Baseline] Epoch 4: loss = 0.6377
[Baseline] Epoch 5: loss = 0.6263
[Baseline] Epoch 6: loss = 0.6208
[Baseline] Epoch 7: loss = 0.6161
[Baseline] Epoch 8: loss = 0.6103
[Baseline] Epoch 9: loss = 0.6097
[Baseline] Epoch 10: loss = 0.6054
[Baseline] Epoch 11: loss = 0.5995
[Baseline] Epoch 12: loss = 0.6077
[Baseline] Epoch 13: loss = 0.6042
[Baseline] Epoch 14: loss = 0.5948
[Baseline] Epoch 15: loss = 0.5982
[Baseline] Epoch 16: loss = 0.6075
[Baseline] Epoch 17: loss = 0.5987
[Baseline] Epoch 18: loss = 0.5965
[Baseline] Epoch 19: loss = 0.5964
[Baseline] Epoch 20: loss = 0.5925


In [54]:


def edit_weighted_loss(
    logits,
    targets,
    edit_mask,
    pad_id,
    alpha=1.0
):
    """
    logits: (B, T, V)
    targets: (B, T)
    edit_mask: (B, T), 0 or 1
    """

    B, T, V = logits.shape

    # Token-level CE
    ce = F.cross_entropy(
        logits.reshape(-1, V),
        targets.reshape(-1),
        ignore_index=pad_id,
        reduction="none"
    ).view(B, T)

    # Weight edited positions
    weights = 1.0 + alpha * edit_mask.float()

    # Mask out PAD positions explicitly
    valid = (targets != pad_id).float()

    loss = (ce * weights * valid).sum() / valid.sum()
    return loss


def train_edit_aware(
    model,
    dataloader,
    optimizer,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    alpha=1.0,
    epochs=10
):
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for corrupted, clean, edit_mask, _ in dataloader:
            corrupted = corrupted.to(device)
            clean = clean.to(device)
            edit_mask = edit_mask.to(device)

            inputs = corrupted[:, :-1]
            targets = clean[:, 1:]
            mask = edit_mask[:, 1:]

            logits = model(inputs)

            loss = edit_weighted_loss(
                logits,
                targets,
                mask,
                pad_id=PAD_ID,
                alpha=alpha
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"[Edit-aware α={alpha}] Epoch {epoch+1}: loss = {avg_loss:.4f}")


In [55]:
edit_aware_model = MelodyTransformer(vocab_size=len(vocab), d_model=256, n_heads=4, n_layers=4)
optimizer = torch.optim.Adam(edit_aware_model.parameters(), lr=0.0001)

In [56]:
train_edit_aware(edit_aware_model, train_dataloader, optimizer, epochs=2)

ValueError: Expected input batch_size (4192) to match target batch_size (4128).